# Dev notebook for generic volume data

## Imports

In [ ]:
import numpy as np
from typing import Tuple
from matplotlib import pyplot as plt
from skimage import measure
from PIL import Image
import numpy.lib.recfunctions as rf
import os
os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.9\bin")
from _preprocess_module import HeightFieldExtractor



## Python class to represent generic volume data

In [ ]:
class GenericVolumeContainer:
    def __init__(self, shape: Tuple[int, int, int]):
        self.size_x, self.size_y, self.size_z = shape
        self.container = np.zeros(shape, dtype=np.float32)

    def add_cuboid(self, shape: Tuple[int, int, int], coord: Tuple[int, int, int]):
            s_x, s_y, s_z = shape
            c_x, c_y, c_z = coord
            for x in range(s_x):
                for y in range(s_y):
                    for z in range(s_z):
                        self.set_value_at_coord((c_x + x,c_y + y,c_z + z), 100.0)

    def get_value_at_coord(self, coord: Tuple[int, int, int]) -> np.float32:
        if not self.is_coord_valid(coord):
            raise IndexError
        x,y,z = coord
        return self.container[x][y][z]


    def set_value_at_coord(self, coord: Tuple[int, int, int], value: float) -> None:
        if self.is_coord_valid(coord):
            x, y, z = coord
            self.container[x][y][z] = np.float32(value)


    def is_coord_valid(self, coord: Tuple[int, int, int]) -> bool:
        x,y,z = coord
        return self.check_x_pos(x) and self.check_y_pos(y) and self.check_z_pos(z)
       
    
    def check_x_pos(self, x: int) -> bool:
        return x >= 0 and x < self.size_x

    def check_y_pos(self, y: int) -> bool:
        return y >= 0 and y < self.size_y

    def check_z_pos(self, z: int) -> bool:
        return z >= 0 and z < self.size_z


    def add_sphere(self, radius: int, coord: Tuple[int, int,  int], edge_width: int = 1) -> None:
        coord_x, coord_y, coord_z = coord

        x = np.arange(self.container.shape[0])[:, None, None]
        y = np.arange(self.container.shape[1])[None, : , None]
        z = np.arange(self.container.shape[2])[None, None, :]

        distance = np.sqrt(((x-coord_x) ** 2) + ((y-coord_y) ** 2) + ((z-coord_z) ** 2))
        signed_distance = radius - distance

        sphere_values = np.clip(signed_distance/edge_width, 0.0, 1.0) * 100.0

        self.container = np.maximum(self.container, sphere_values.astype(np.float32))
    

In [ ]:
PRINT_COORD_VALS = False
PLOT_IMG = False

def visualize_container(volume: GenericVolumeContainer) -> None:
    fig = plt.figure()
    ax = fig.add_subplot(projection="3d")
    ax.voxels(volume.container)
    ax.set_xlim(0, volume.size_x)
    ax.set_ylim(0, volume.size_y)
    ax.set_zlim(0, volume.size_z)
    plt.show()

def print_volume_values_running_z(volume: GenericVolumeContainer, x: int, y: int, z_start: int, z_stop:int) -> None:
    print(f"Values for x: {x}, y: {y}, running z from {z_start} to {z_stop}")
    for i in range(z_start, z_stop):
        print(f"Value at z {i}: {volume.get_value_at_coord((x,y,i))}")

def test_generic_volume():
    volume = GenericVolumeContainer((50,50,50))
    volume.add_cuboid((10,10,10), (20,20,20))
    volume.add_sphere(10, (10,10,10))
    if PRINT_COORD_VALS:
        print_volume_values_running_z(volume, 19, 10, 0, 20)
    if PLOT_IMG:
        visualize_container(volume)
    

In [ ]:
test_generic_volume()

## Interfaces needed to transfer the volume data to cpp

### add_volume

- [x] shape information to allocate a matrix
- [x] copy the matrix

### intersect
- [x] Extract the extended heighfield using texture memory
- [ ] Extract the normal map

## Notes

- Result of the heightfield is faulty. The result is copied 4 times in the image.

In [ ]:
vol = GenericVolumeContainer((1000,1000,500))
vol.add_cuboid((120,120,120), (20,20,20))
vol.add_sphere(50, (900,900,400))


preprocessor = HeightFieldExtractor( (1000,1000), 2, 256 )
preprocessor.add_volume(vol.container, 1000, 1000, 500)

extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

# Save the extended heightfields
for z in range(extended_heightfield.shape[2]):
    # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
    entry = extended_heightfield[:,:,z]
    entry = entry.astype(np.uint16)
    img = Image.fromarray(entry, "I;16")
    img.save("output/integrated_"+str(z)+".tif")

# Convert the first normal map
normal_map = normal_map.squeeze(2)
normal_map = rf.structured_to_unstructured( normal_map )
normal_map = ( normal_map + 1.0 ) * 127.5
normal = normal_map.astype(np.uint8)
del preprocessor